<a id="activity"></a>
# VideoDB Understanding: Activity

Turn a video into a timeline of actions and goals with the `activity` analyzer, then click through it and watch any moment.

The sample is a first-person video of a person folding laundry, uploaded straight from its URL.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/indexing-v2/understanding/activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a id="install"></a>
## 1. Install and connect

Enter your VideoDB API key when prompted, or set `VIDEO_DB_API_KEY` in your environment or `.env`.

In [ ]:
!pip install -q "videodb>=0.5.0" python-dotenv pandas

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()
print("Connected to VideoDB")
print("Collection:", collection.id)

<a id="helpers"></a>
## 2. Helpers

Everything that renders or plays results lives here, so the rest of the notebook only calls it. The code is collapsed by default; run the three cells and move on.

| Helper | What it does |
|---|---|
| `activity_tables(output)` | Splits the analyzer output into `actions`, `goals` and `warnings` |
| `show_summary(...)` | One-line overview plus any processing warnings |
| `show_explorer(video, actions)` | Clickable goal timeline, per-goal action waterfall and a video player |
| `clip_goal(...)` / `clip_action(...)` | Make a standalone stream of just that moment |

In [ ]:
#@title Data helpers: activity_tables (run this cell; double-click to view the code)
import json
import uuid
from html import escape

import pandas as pd
from IPython.display import HTML, display
from videodb import play_stream


def clock(seconds):
    """Format seconds as m:ss."""
    if pd.isna(seconds):
        return "–"
    minutes, secs = divmod(int(round(seconds)), 60)
    return f"{minutes}:{secs:02d}"


def plain(value):
    """Turn a table value into something JSON can hold; missing becomes None."""
    if pd.isna(value):
        return None
    return value.item() if hasattr(value, "item") else value


def activity_tables(output):
    """Split Activity output into (actions, goals, warnings)."""
    metadata = output.get("metadata") or {}
    rows = []
    for scene in output.get("scenes") or []:
        data = scene.get("data") or {}
        rows.append({
            "scene_id": scene.get("scene_id"),
            "start": scene.get("start"),
            "end": scene.get("end"),
            "action": data.get("instruction"),
            "goal": data.get("goal_instruction"),
            "confidence": data.get("confidence"),
            "issue": data.get("issue"),
        })
    actions = pd.DataFrame(
        rows, columns=["scene_id", "start", "end", "action", "goal", "confidence", "issue"]
    )
    goals = pd.DataFrame(metadata.get("goals") or [])
    return actions, goals, metadata.get("warnings") or []


def goal_groups(actions):
    """Group timed actions by goal, in order of first appearance: [(goal, its actions)]."""
    timed = actions.dropna(subset=["start", "end"])
    return list(timed.groupby(timed["goal"].fillna("No goal"), sort=False))

In [ ]:
#@title Display helpers: summary and explorer (run this cell; double-click to view the code)
GOAL_COLORS = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756", "#72B7B2", "#EECA3B", "#9D755D"]


def show_summary(actions, goals, warnings):
    """Show what the run produced and anything that needs review."""
    timed = actions.dropna(subset=["start", "end"])
    span = f"{clock(timed['start'].min())}–{clock(timed['end'].max())}" if len(timed) else "no timed actions"
    flagged = int(actions["issue"].notna().sum())
    parts = [f"<b>{len(actions)}</b> actions", f"<b>{len(goals)}</b> goals", span]
    if flagged:
        parts.append(f"<b>{flagged}</b> with an issue")
    html = f"<p style='font-size:15px'>{' &nbsp;·&nbsp; '.join(parts)}</p>"
    if warnings:
        items = "".join(f"<li>{escape(str(warning))}</li>" for warning in warnings)
        html += f"<p><b>Processing warnings</b>: review before relying on the timeline.</p><ul>{items}</ul>"
    display(HTML(html))


EXPLORER_TEMPLATE = """
<style>
#__ID__ { max-width: 1150px; font-size: 13px; }
#__ID__ .top { display: flex; gap: 18px; flex-wrap: wrap; align-items: flex-start; }
#__ID__ .pane { flex: 1 1 440px; min-width: 0; }
#__ID__ .player { flex: 0 1 430px; }
#__ID__ video { width: 100%; aspect-ratio: 16/9; background: #000; border-radius: 6px; display: block; }
#__ID__ .now { margin-top: 6px; min-height: 2.6em; opacity: .85; }
#__ID__ .hd { font-weight: 600; margin: 10px 0 4px; }
#__ID__ .list { overflow-y: auto; }
#__ID__ .goals { max-height: 300px; }
#__ID__ .actions { max-height: 300px; }
#__ID__ .row { display: flex; align-items: center; gap: 8px; padding: 3px 6px; border-radius: 4px; cursor: pointer; }
#__ID__ .row:hover { background: rgba(128,128,128,.15); }
#__ID__ .row.sel, #__ID__ .row.on { background: rgba(128,128,128,.3); }
#__ID__ .num { flex: none; width: 22px; text-align: right; opacity: .55; }
#__ID__ .time { flex: none; width: 82px; opacity: .75; font-variant-numeric: tabular-nums; }
#__ID__ .label { flex: 0 0 40%; overflow: hidden; white-space: nowrap; text-overflow: ellipsis; }
#__ID__ .text { flex: 0 0 45%; }
#__ID__ .track { position: relative; flex: 1; min-width: 60px; height: 18px; border-radius: 3px; background: rgba(128,128,128,.15); }
#__ID__ .bar { position: absolute; top: 2px; bottom: 2px; border-radius: 2px; }
#__ID__ .bar.issue { outline: 2px dashed #d62728; }
#__ID__ .conf { flex: none; width: 40px; text-align: right; opacity: .75; }
</style>
<div id="__ID__">
  <div class="top">
    <div class="pane">
      <div class="hd">Goals · click one to watch it</div>
      <div class="list goals"></div>
    </div>
    <div class="player">
      <video controls playsinline></video>
      <div class="now">Click a goal or an action to play it.</div>
    </div>
  </div>
  <div class="hd actions-title"></div>
  <div class="list actions"></div>
</div>
<script>
(function () {
  const data = __DATA__;
  const root = document.getElementById("__ID__");
  const video = root.querySelector("video");
  const goalsEl = root.querySelector(".goals");
  const actionsEl = root.querySelector(".actions");
  const titleEl = root.querySelector(".actions-title");
  const nowEl = root.querySelector(".now");
  let stopAt = null;
  let selected = null;

  const clock = (t) => Math.floor(Math.round(t) / 60) + ":" + String(Math.round(t) % 60).padStart(2, "0");
  const range = (item) => clock(item.start) + "–" + clock(item.end);
  const el = (cls, text) => {
    const node = document.createElement("div");
    node.className = cls;
    if (text != null) node.textContent = text;
    return node;
  };
  const bar = (item, from, span, color) => {
    const node = el("bar" + (item.issue ? " issue" : ""));
    node.style.left = (100 * (item.start - from)) / span + "%";
    node.style.width = Math.max((100 * (item.end - item.start)) / span, 0.5) + "%";
    node.style.background = color;
    return node;
  };

  function play(item, label) {
    stopAt = item.end;
    nowEl.textContent = range(item) + "  " + label;
    const go = () => { video.currentTime = item.start; video.play(); };
    if (video.readyState >= 1) go();
    else video.addEventListener("loadedmetadata", go, { once: true });
  }

  function select(index, autoplay) {
    selected = data.goals[index];
    [...goalsEl.children].forEach((row, i) => row.classList.toggle("sel", i === index));
    titleEl.textContent = "Actions in “" + selected.label + "” · click one to watch it";
    const span = Math.max(selected.end - selected.start, 0.001);
    actionsEl.replaceChildren(...selected.actions.map((action) => {
      const row = el("row");
      const track = el("track");
      track.append(bar(action, selected.start, span, selected.color));
      row.append(el("num", action.row), el("time", range(action)), el("text", action.text), track,
                 el("conf", action.confidence == null ? "" : action.confidence));
      if (action.issue) row.title = "Issue: " + action.issue;
      row.onclick = () => play(action, action.text);
      return row;
    }));
    if (autoplay) play(selected, selected.label);
  }

  data.goals.forEach((goal, index) => {
    const row = el("row");
    const track = el("track");
    goal.actions.forEach((action) => track.append(bar(action, 0, data.total, goal.color)));
    const label = el("label", goal.label);
    label.title = goal.label;
    row.append(el("num", index), label, track, el("time", range(goal)));
    row.onclick = () => select(index, true);
    goalsEl.append(row);
  });

  video.addEventListener("timeupdate", () => {
    const now = video.currentTime;
    if (stopAt != null && now >= stopAt) { video.pause(); stopAt = null; }
    if (selected) [...actionsEl.children].forEach((row, i) => {
      const action = selected.actions[i];
      row.classList.toggle("on", now >= action.start && now < action.end);
    });
  });

  const attach = () => {
    if (window.Hls && window.Hls.isSupported()) {
      const hls = new window.Hls();
      hls.loadSource(data.stream);
      hls.attachMedia(video);
    } else {
      video.src = data.stream;
    }
  };
  if (window.Hls) attach();
  else {
    const script = document.createElement("script");
    script.src = "https://cdn.jsdelivr.net/npm/hls.js@1";
    script.onload = attach;
    script.onerror = attach;
    document.head.append(script);
  }
  select(0, false);
})();
</script>
"""


def show_explorer(video, actions):
    """Clickable results: goals on a timeline, the selected goal's actions, and a player.

    Clicking plays that span of the video's own stream, so nothing is re-rendered.
    """
    groups = goal_groups(actions)
    if not groups:
        print("No timed actions to show.")
        return
    goals = [{
        "label": goal,
        "start": float(group["start"].min()),
        "end": float(group["end"].max()),
        "color": GOAL_COLORS[lane % len(GOAL_COLORS)],
        "actions": [{
            "row": int(row),
            "start": float(action["start"]),
            "end": float(action["end"]),
            "text": plain(action["action"]) or "",
            "confidence": plain(action["confidence"]),
            "issue": plain(action["issue"]),
        } for row, action in group.iterrows()],
    } for lane, (goal, group) in enumerate(groups)]
    data = {
        "stream": video.generate_stream(),
        "total": max(goal["end"] for goal in goals) or 1.0,
        "goals": goals,
    }
    html = EXPLORER_TEMPLATE.replace("__ID__", f"activity-{uuid.uuid4().hex[:8]}")
    display(HTML(html.replace("__DATA__", json.dumps(data).replace("</", "<\\/"))))

In [ ]:
#@title Clip helpers: clip_goal, clip_action (run this cell; double-click to view the code)
def clip(video, start, end, label=""):
    """Generate a standalone stream of one interval and play it. Does not rerun Activity."""
    stream_url = video.generate_stream(timeline=[(float(start), float(end))])
    print(f"{clock(start)}–{clock(end)}  {label}")
    print("Stream URL:", stream_url)
    return play_stream(stream_url)


def clip_goal(video, actions, goal):
    """Clip a whole goal. `goal` is its number in the explorer's goal list."""
    label, group = goal_groups(actions)[goal]
    return clip(video, group["start"].min(), group["end"].max(), label)


def clip_action(video, actions, row):
    """Clip one action. `row` is its number in the explorer's action list."""
    action = actions.loc[row]
    if pd.isna(action["start"]) or pd.isna(action["end"]):
        raise ValueError("This action has no time interval; choose another.")
    return clip(video, action["start"], action["end"], action["action"])

<a id="video"></a>
## 3. Upload a video

VideoDB fetches the video from the URL directly; no local download is needed. Set `VIDEODB_VIDEO_URL` to try your own.

In [ ]:
VIDEO_URL = os.getenv(
    "VIDEODB_VIDEO_URL",
    "https://www.youtube.com/watch?v=wuRf25_zME0",  # First-person laundry folding
)

video = collection.upload(url=VIDEO_URL)

# To reuse a video already in your account, replace the upload above with:
# video = collection.get_video("m-...")

print("Video:", video.id)
video.play()

<a id="run"></a>
## 4. Run Activity

Activity prepares the frames and runs both analysis passes for you, using the server's default model. Pick a mode:

- **`express`** starts processing immediately.
- **`batch`** saves 50% on model inference costs, but can take longer.

Each run of this cell creates a new Understanding.

In [ ]:
MODE = "express"  # or "batch"

understanding = video.understand(
    analyzers=[{"type": "activity", "name": "activity", "config": {"mode": MODE}}],
)
print("Understanding ID:", understanding.id)

<a id="wait"></a>
## 5. Wait for results

If this times out, rerun this cell: it keeps waiting on the same Understanding.

In [ ]:
understanding.wait_until_complete(timeout=48 * 3600, poll_interval=60)

analyzer = understanding.get_analyzer("activity", refresh=True)
if not analyzer.is_successful:
    raise RuntimeError(f"Activity did not succeed: {analyzer.status}")

actions, goals, warnings = activity_tables(analyzer.get_output())
print("Status:", analyzer.status)

<a id="results"></a>
## 6. Explore the results

- **Goals**: one lane per goal, one block per action. Click a goal to watch it.
- **Actions**: the selected goal's actions, zoomed to that goal, with confidence on the right. Click one to watch just that action; the row that is playing stays highlighted.
- A dashed red outline marks an action with an issue; hover its row to read it.

In [ ]:
show_summary(actions, goals, warnings)
show_explorer(video, actions)

<a id="clip"></a>
## 7. Share a clip

The explorer plays spans of the full video. To hand someone a moment on its own, generate a stream for it: use the numbers shown beside each goal and action. The printed URL works in any HLS player.

In [ ]:
clip_goal(video, actions, goal=0)

In [ ]:
clip_action(video, actions, row=0)

`actions` and `goals` are ordinary DataFrames, so you can filter or export them. `goals` holds the analyzer's own goal records, including `goal_id` and `uncertainty`.

<a id="next"></a>
## Next steps

| Goal | Notebook |
|---|---|
| Explore every analyzer | [Understanding guide map](README.md) |
| Manage runs, statuses and callbacks | [Outputs and operations](outputs-and-operations.ipynb) |
| Compose dependent analyzers | [Multi-analyzer pipelines](multi-analyzer-pipelines.ipynb) |